<a target="_blank" rel="noopener noreferrer" href="https://colab.research.google.com/github/ccaudek/ds4psy_2023/blob/main/kl.ipynb">![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)</a>

(kl_notebook)=
# La divergenza di Kullback-Leibler

```{admonition} Obiettivi di apprendimento
Dopo aver completato questo capitolo, acquisirete le competenze per:

- comprendere il concetto di divervenza di Kullback-Leibler (KL);
- calcolare la divergenza KL dall'entropia;
- comprendere il concetto della Densità Logaritmica Predittiva Prevista (ELPD);
- mettere in relazione il concetto di entropia con la ELPD;
- calcolare la ELPD con PyMC.
```

Immaginate di avere una ricetta complessa, con molti ingredienti e passaggi. Questa ricetta rappresenta una distribuzione di probabilità complessa, chiamata $p$. A volte, questa ricetta è troppo complicata per essere seguita esattamente. Quindi, usiamo una versione più semplice, una ricetta "approssimativa", chiamata $q$.

In statistica e machine learning, spesso incontriamo situazioni simili. La distribuzione $p$ potrebbe essere troppo complessa o sconosciuta, quindi usiamo $q$ come una versione semplificata. Però, ci chiediamo: quanto è "buona" questa versione semplificata rispetto all'originale?

Per valutare quanto bene la nostra ricetta approssimativa $q$ si avvicina alla ricetta originale $p$, usiamo qualcosa chiamato *Divergenza di Kullback-Leibler* ($\mathbb{KL}$). Questo è come confrontare il gusto dei piatti preparati con le due ricette diverse.

Se la Divergenza $\mathbb{KL}$ è zero, significa che le due ricette producono esattamente lo stesso piatto. Valori maggiori di zero indicano differenze maggiori tra i piatti. Quindi, un valore basso di $\mathbb{KL}$ significa che la nostra ricetta approssimativa è molto vicina all'originale.

Un altro strumento importante è l'*Expected Log Predictive Density* (elpd), che è come valutare quanto bene una ricetta si adatta ai gusti di diversi ospiti. L'elpd guarda quanto bene una ricetta (modello statistico) si adatta ai dati (gusti degli ospiti), bilanciando la semplicità della ricetta con la sua capacità di soddisfare tutti.

Un modello che si adatta perfettamente ai dati ma è troppo complicato potrebbe non essere pratico, proprio come una ricetta troppo elaborata. Allo stesso modo, una ricetta troppo semplice potrebbe non soddisfare tutti i gusti. L'elpd ci aiuta a trovare il giusto equilibrio.

In questo capitolo, ci concentreremo sulla Divergenza $\mathbb{KL}$ e l'elpd. Esploreremo come questi strumenti ci aiutano a valutare e confrontare modelli statistici, specialmente in contesti bayesiani. Comprendendo bene $\mathbb{KL}$ e elpd, potrete scegliere e analizzare modelli statistici più efficacemente, adattandoli ai dati specifici che avete.

In [1]:
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.graphics import tsaplots
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import pymc.sampling_jax
from scipy.stats import beta
from scipy.integrate import quad
import arviz as az

In [2]:
%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-grayscale")
from cycler import cycler
default_cycler = cycler(color=["#000000", "#6a6a6a", "#bebebe", "#2a2eec"])
plt.rc('axes', prop_cycle=default_cycler)

## Una Misura della Perdita di Informazione

La Divergenza di Kullback-Leibler ($\mathbb{KL}$), conosciuta anche come entropia relativa, funge da misura quantitativa della perdita di informazione che si verifica quando una distribuzione di probabilità complessa o sconosciuta, denominata $p$, viene approssimata con una distribuzione più semplice, chiamata $q$. Questa misura ci fornisce una valutazione di quanto le due distribuzioni differiscano l'una dall'altra in termini informativi.

Dal punto di vista matematico, la $\mathbb{KL}$ si calcola sommando le differenze tra le probabilità logaritmiche associate ad ogni evento possibile secondo $p$ e $q$, ponderate per le probabilità degli eventi secondo $p$. La formula generale è:

$$
\mathbb{KL}(p \parallel q) = \sum_{i=1}^n p_i (\log p_i - \log q_i),
$$

dove $i$ indica ciascun possibile evento all'interno delle distribuzioni. Questo calcolo produce una misura della "distanza" media tra $p$ e $q$, considerando le loro probabilità logaritmiche.

La Divergenza $\mathbb{KL}$ può anche essere interpretata attraverso il concetto di entropia, che rappresenta la misura dell'incertezza o della variabilità intrinseca a una distribuzione di probabilità. In questo contesto, si confrontano due elementi:

1. **Entropia Incrociata ($H(p, q)$)**: Questa misura riflette quanto è "disordinata" o imprevedibile la distribuzione $q$ quando analizzata attraverso le lenti delle probabilità fornite da $p$.

   $$
   H(p, q) = -\sum_{i=1}^n p_i \log q_i.
   $$

2. **Entropia di $p$ ($H(p)$)**: Questa rappresenta l'incertezza intrinseca alla distribuzione originale $p$.

   $$
   H(p) = -\sum_{i=1}^n p_i \log p_i.
   $$

Conseguentemente, la Divergenza $\mathbb{KL}$ emerge come la differenza tra l'entropia incrociata e l'entropia di $p$:

$$
\begin{align}
\mathbb{KL}(p \parallel q) &= H(p, q) - H(p) \notag\\
&= -\sum_{i=1}^n p_i \log q_i - (-\sum_{i=1}^n p_i \log p_i) \notag\\
&= \sum_{i=1}^n p_i \log p_i -\sum_{i=1}^n p_i \log q_i \notag\\
&= \sum_{i=1}^n p_i (\log p_i - \log q_i). \notag
\end{align}
$$

Questa differenza indica in che misura l'utilizzo di $q$ come approssimazione di $p$ ci porta a deviare dalla "realtà" rappresentata da $p$. Una $\mathbb{KL}$ bassa suggerisce che $q$ è un'approssimazione efficace di $p$, con una minima perdita di informazione. Al contrario, un valore elevato di $\mathbb{KL}$ segnala una sostanziale perdita di informazione, indicando che $q$ potrebbe non essere adeguata per rappresentare fedelmente $p$.

### Un Esempio Empirico

Per comprendere meglio questi concetti, esaminiamo ora un esempio numerico, dove definiremo due distribuzioni di probabilità discrete, $p$ e $q$.

In [45]:
p = np.array([0.2, 0.5, 0.3])
q = np.array([0.1, 0.2, 0.7])

Calcoliamo l'entropia di $p$.

In [46]:
h_p = -np.sum(p * np.log(p))
print("Entropia di p: ", h_p)

Entropia di p:  1.0296530140645737


Calcoliamo l'entropia incrociata tra $p$ e $q$.

In [47]:
h_pq = -np.sum(p * np.log(q))
print("Entropia incrociata tra p e q: ", h_pq)

Entropia incrociata tra p e q:  1.372238457997479


Calcoliamo la divergenza di Kullback-Leibler da $p$ a $q$.

In [48]:
kl_pq = h_pq - h_p
print("Divergenza KL da p a q: ", kl_pq)

Divergenza KL da p a q:  0.34258544393290524


Lo stesso risultato si ottiene applicando la formula della Divergenza $\mathbb{KL}$.

In [52]:
np.sum(p * (np.log(p) - np.log(q)))

0.3425854439329054

Se invece $q$ è molto simile a $p$, la differenza $\mathbb{KL}$ è molto minore.

In [53]:
p = np.array([0.2, 0.5, 0.3])
q = np.array([0.2, 0.55, 0.25])
np.sum(p * (np.log(p) - np.log(q)))

0.007041377136023895

Consideriamo un secondo esempio. Sia $p$ una distribuzione binomiale di parametri $\theta = 0.2$ e $n = 5$.

In [10]:
# Define the parameters
n = 4
p = 0.2

# Compute the probability mass function
true_py = stats.binom.pmf(range(n + 1), n, p)
print(true_py)

[0.4096 0.4096 0.1536 0.0256 0.0016]


Sia $q_1$ una approssimazione a $p$:

In [12]:
q1 = np.array([0.46, 0.42, 0.10, 0.01, 0.01])
print(q1)

[0.46 0.42 0.1  0.01 0.01]


Sia $q_2$ una distribuzione uniforme:

In [14]:
q2 = [0.2] * 5
print(q2)

[0.2, 0.2, 0.2, 0.2, 0.2]


La divergenza $\mathbb{KL}$ di $q_1$ da $p$ è

In [19]:
kl_pq1 = np.sum(true_py * (np.log(true_py) - np.log(q1)))
print("Divergenza KL di q1 da p: ", kl_pq1)

Divergenza KL di q1 da p:  0.02925199033345885


La divergenza $\mathbb{KL}$ di $q_2$ da $p$ è:

In [20]:
kl_pq2 = np.sum(true_py * (np.log(true_py) - np.log(q2)))
print("Divergenza KL di q2 da p: ", kl_pq2)

Divergenza KL di q2 da p:  0.4863577787141543


È chiaro che perdiamo una quantità maggiore di informazioni se, per descrivere la distribuzione binomiale $p$, usiamo la distribuzione uniforme $q_2$ anziché $q_1$.

### La Divergenza Dipende dalla Direzione

La Divergenza $\mathbb{KL}$ è spesso paragonata a una "distanza" tra due distribuzioni di probabilità, ma è fondamentale capire che non è simmetrica. Questo significa che la misura di quanto $p$ è diversa da $q$ non è la stessa di quanto $q$ è diversa da $p$. Questa asimmetria riflette la differenza nella perdita di informazione quando si sostituisce una distribuzione con l'altra.

Per comprendere meglio la Divergenza $\mathbb{KL}$, è utile considerare l'entropia e l'entropia incrociata. Facciamo un esempio numerico.

In [55]:
# Definire le distribuzioni p e q
p = np.array([0.01, 0.99])
q = np.array([0.7, 0.3])

# Calcolo dell'entropia di p
h_p = -np.sum(p * np.log(p))

# Calcolo dell'entropia incrociata da p a q
h_pq = -np.sum(p * np.log(q))

# Calcolo della divergenza KL da p a q
kl_pq = h_pq - h_p

# Calcolo dell'entropia di q
h_q = -np.sum(q * np.log(q))

# Calcolo dell'entropia incrociata da q a p
h_qp = -np.sum(q * np.log(p))

# Calcolo della divergenza KL da q a p
kl_qp = h_qp - h_q

print(f"Entropia di p: {h_p}")
print(f"Entropia incrociata da p a q: {h_pq}")
print(f"Divergenza KL da p a q: {kl_pq}")

print(f"\nEntropia di q: {h_q}")
print(f"Entropia incrociata da q a p: {h_qp}")
print(f"Divergenza KL da q a p: {kl_qp}")

Entropia di p: 0.056001534354847345
Entropia incrociata da p a q: 1.1954998257220641
Divergenza KL da p a q: 1.1394982913672167

Entropia di q: 0.6108643020548935
Entropia incrociata da q a p: 3.226634230947714
Divergenza KL da q a p: 2.6157699288928207


- **Entropia di $p$ ($h(p)$)**: Misura l'incertezza o la variabilità all'interno della distribuzione vera $p$. Nel nostro esempio, l'entropia di $p$ è 0.056.
- **Entropia Incrociata da $p$ a $q$ ($h(p, q)$)**: Misura l'incertezza quando si usa $q$ per rappresentare $p$. Nel nostro esempio, è 1.195.

La Divergenza $\mathbb{KL}$ da $p$ a $q$ si calcola come la differenza tra l'entropia incrociata e l'entropia di $p$, che nel nostro caso è 1.139. Questo valore indica la perdita di informazione quando si utilizza $q$ per approssimare $p$.

Quando invertiamo le distribuzioni e calcoliamo la Divergenza $\mathbb{KL}$ da $q$ a $p$, otteniamo un risultato diverso. L'entropia di $q$ è 0.611 e l'entropia incrociata da $q$ a $p$ è 3.227. La Divergenza $\mathbb{KL}$ risultante da $q$ a $p$ è 2.616, molto più grande rispetto a quella da $p$ a $q$.

Questi risultati dimostrano chiaramente che la Divergenza $\mathbb{KL}$ è asimmetrica. La quantità di informazione che si perde nel sostituire $p$ con $q$ non è la stessa che si perde sostituendo $q$ con $p$. Questa asimmetria è una caratteristica cruciale della Divergenza $\mathbb{KL}$ e sottolinea l'importanza di considerare attentamente quale distribuzione si sta utilizzando come approssimazione dell'altra.

## Confronto di Modelli Utilizzando la Divergenza di Kullback-Leibler

La Divergenza $\mathbb{KL}$ è uno strumento essenziale per confrontare diversi modelli statistici. Questo metodo misura la differenza tra la distribuzione di probabilità teorizzata da un modello, che chiamiamo $p_{\mathcal{M}}$, e la distribuzione di probabilità che ha effettivamente generato i dati, denominata $p_t$.

### Distribuzione Predittiva Posteriori

Abbiamo introdotto il concetto di distribuzione predittiva posteriori, definita come:

$$
p(\tilde{y} \mid y) = \int_\Theta p(\tilde{y} \mid \theta) p(\theta \mid y) \, \mathrm{d}\theta .
$$

Questa formula rappresenta la previsione del tipo di dati che il modello $\mathcal{M}$ potrebbe generare. Questa previsione è basata sulle nostre conoscenze precedenti, espresse come $p(\theta)$, e sui dati osservati, $y$.

### Misurare la Somiglianza tra Distribuzioni

L'obiettivo è valutare quanto la distribuzione $q_{\mathcal{M}} = p(\tilde{y} \mid y)$ si avvicini alla distribuzione del modello generatore reale $p_t(\tilde{y})$. In altre parole, cerchiamo di quantificare la similitudine tra i dati prodotti dal modello teorico $q_{\mathcal{M}}$ e quelli generati dal modello reale $p_t$. Questo viene fatto attraverso la Divergenza di Kullback-Leibler:

$$
\mathbb{KL}(p_t \mid\mid q_{\mathcal{M}}).
$$

### Confrontare Più Modelli

Supponiamo di avere una serie di $k$ modelli distinti $\{q_{\mathcal{M}_1}, q_{\mathcal{M}_2}, \ldots, q_{\mathcal{M}_k}\}$. Se conoscessimo $p_t$, potremmo calcolare la Divergenza KL per ogni modello in questo modo:

$$
\begin{align*}
\mathbb{KL} (p_t \mid\mid q_{\mathcal{M}_1}) &= \mathbb{E} (\log p_t) - \mathbb{E} (\log q_{\mathcal{M}_1}), \\
\mathbb{KL} (p_t \mid\mid q_{\mathcal{M}_2}) &= \mathbb{E} (\log p_t) - \mathbb{E} (\log q_{\mathcal{M}_2}), \\
&\vdots \\
\mathbb{KL} (p_t \mid\mid q_{\mathcal{M}_k}) &= \mathbb{E} (\log p_t) - \mathbb{E} (\log q_{\mathcal{M}_k}).
\end{align*}
$$ (eq-kl-mod-comp)

In pratica, però, $p_t$ è sconosciuta. Fortunatamente, $p_t$ rimane costante in tutte le equazioni, permettendoci di confrontare i modelli focalizzandoci sul secondo termine della Divergenza KL, che è indipendente da $p_t$. Per un modello generico $\mathcal{M}$, questo termine è:

$$
\mathbb{E} \log p_{\mathcal{M}}(y) = \int_{-\infty}^{+\infty} p_t(y) \log p_{\mathcal{M}}(y) \, \mathrm{d}y .
$$ (eq-kl-div-cont-t2)

In questo modo, possiamo costruire un metodo per confrontare i modelli senza dover conoscere il modello generatore effettivo dei dati.

In conclusione, l'eq. {eq}`eq-kl-div-cont-t2` calcola la densità logaritmica media della distribuzione $p_{\mathcal{M}}(y)$ basata sulla distribuzione reale $p_t(y)$. Ciò indica quanto accuratamente il modello $\mathcal{M}$ rappresenta i dati reali generati da $p_t$.

## Considerazioni Conclusive

La divergenza di Kullback-Leibler (KL) è una misura di dissimilarità tra due distribuzioni di probabilità. Nel contesto della valutazione dei modelli, la divergenza KL viene spesso usata per quantificare quanto bene un modello probabilistico approssima la distribuzione vera dei dati. Specificamente, valuta quanto bene la distribuzione predittiva del modello (q) si avvicina alla distribuzione reale dei dati ($p$). In questo senso, la divergenza KL è più focalizzata sull'adattamento del modello ai dati osservati.

Se invece vogliamo un criterio che valuta la capacità di un modello di fare previsioni su nuovi dati, non ancora osservati, allora dobbiamo usare l'Expected Log Predictive Density (ELPD), un concetto che verrà introdotto nel capitolo successivo. L'ELPD misura, in termini di log-verosimiglianza, quanto bene il modello si aspetta di predire nuovi dati, riflettendo la capacità di generalizzazione del modello. In sintesi, mentre la divergenza KL si concentra sull'adattamento ai dati osservati, l'ELPD valuta la capacità di generalizzazione a nuovi dati.

Dobbiamo però riconoscere che la vera densità dei dati ($p$) è sconosciuta, il che rende la stima diretta dell'ELPD impraticabile. Per valutare la capacità di generalizzazione dei modelli, dobbiamo quindi affidarci a metodi di approssimazione. Ci sono diversi approcci per approssimare l'ELPD, tra cui:

- **Leave-One-Out Cross-Validation (LOO-CV)**: Questo metodo è ampiamente usato in contesti bayesiani. Consiste nel valutare la capacità di generalizzazione del modello utilizzando una forma di validazione incrociata in cui, per ogni iterazione, un dato viene lasciato fuori dal set di addestramento e il modello viene valutato su quel dato. LOO-CV è particolarmente utile per modelli complessi e campioni di dimensioni ridotte, poiché sfrutta al massimo i dati disponibili.

- **Akaike Information Criterion (AIC)** e **Watanabe-Akaike Information Criterion (WAIC)**: Questi criteri sono basati sui dati osservati e forniscono stime approssimate dell'ELPD. L'AIC è basato sull'informazione di Akaike e cerca di bilanciare la bontà di adattamento del modello con la sua complessità. Il WAIC è una versione più generalizzata dell'AIC pensata per modelli gerarchici e bayesiani, e come l'AIC, mira a quantificare la capacità di generalizzazione del modello.

LOO-CV, AIC, e WAIC sono tutti metodi che cercano di approssimare la capacità di un modello di fare previsioni su nuovi dati, ma attraverso approcci diversi. LOO-CV è più computazionalmente intensivo ma può fornire stime più accurate in certi contesti, mentre AIC e WAIC offrono approcci più rapidi e diretti basati sui dati osservati e sulla struttura del modello.

In sintesi, mentre la divergenza KL si concentra sull'adattamento del modello ai dati attuali, l'ELPD e i suoi metodi di approssimazione (come LOO, AIC, e WAIC) mirano a valutare la capacità di generalizzazione del modello a nuovi dati. Utilizzare queste misure insieme può fornire una visione complessiva della performance di un modello, sia in termini di adattamento che di capacità predittiva.

In [56]:
%load_ext watermark
%watermark -n -u -v -iv -w

Last updated: Thu Jan 25 2024

Python implementation: CPython
Python version       : 3.11.7
IPython version      : 8.19.0

scipy      : 1.11.4
numpy      : 1.26.2
pymc       : 5.10.3
pandas     : 2.1.4
seaborn    : 0.13.0
matplotlib : 3.8.2
statsmodels: 0.14.1
arviz      : 0.17.0

Watermark: 2.4.3

